In [1]:
# Данный ноутбук принимает на вход таблицу формата csv, содержащую следующие колонки:
# Год: int
# Муниципалитет: str
# Пол: str
# Возрастная группа: str
# Причина смерти: str
# Стандартный вес группы: str, должен быть указан в процентах (например '4,4')
# Число умерших: int
# Численность населения: int, имеется в виду численность населения возрастной группы

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import math

In [3]:
# Загрузка исходной таблицы.
df = pd.read_csv("Тестовое _ Данные - Лист1.csv")

# Содержание таблицы

In [4]:
df.head()

,Год,Муниципалитет,Пол,Возрастная группа,Причина смерти,Стандартный вес группы,Число умерших,Численность населения
0,2017,Район 1,Женщины,1-4,Болезни сердца,4,0,155
1,2017,Район 1,Женщины,1-4,Другие болезни органов дыхания,4,0,155
2,2017,Район 1,Женщины,1-4,Другие болезни системы кровообращения,4,0,155
3,2017,Район 1,Женщины,1-4,Острые респираторные заболевания,4,0,155
4,2017,Район 1,Женщины,10-14,Болезни сердца,"5,5",0,163


In [5]:
# Диапазоны значений колонок
print('Диапазоны значений колонок:', '\n')

for column in df.columns:
    print(column, df[column].unique().tolist(), '\n')

Диапазоны значений колонок: 

Год [2017, 2018, 2019, 2020, 2021, 2022, 2023] 

Муниципалитет ['Район 1', 'Район 2', 'Район 3'] 

Пол ['Женщины', 'Мужчины'] 

Возрастная группа ['1-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85+', 'до 1'] 

Причина смерти ['Болезни сердца', 'Другие болезни органов дыхания', 'Другие болезни системы кровообращения', 'Острые респираторные заболевания'] 

Стандартный вес группы ['4', '5,5', '6', '6,5', '7', '5', '2,5', '1'] 

Число умерших [0, 1, 2, 3, 4, 8, 10, 5, 11, 13, 6, 12, 7, 9] 

Численность населения [155, 163, 132, 135, 122, 131, 144, 133, 152, 342, 143, 159, 105, 54, 35, 11, 2, 21, 116, 129, 94, 100, 147, 149, 354, 174, 164, 141, 83, 71, 28, 16, 17, 112, 72, 96, 118, 74, 84, 202, 89, 82, 24, 7, 1, 22, 98, 113, 91, 49, 125, 86, 242, 95, 103, 88, 59, 44, 15, 10, 27, 781, 835, 661, 654, 994, 1223, 1172, 899, 718, 958, 849, 933, 732, 493, 217, 334, 

# Вычисление коэффициентов

## Приведение входных данных к удобному виду

In [6]:
# Приведение колонки 'Стандартный вес группы' исходной таблицы к виду 0.01 (т.е. доли, а не проценты).
df['Стандартный вес группы'] = df['Стандартный вес группы'].str.replace(',', '.').astype(float) / 100

## Функции

In [7]:
def get_slice(
    df: pd.DataFrame,
    district: str,
    year: int,
    target: str = None):

    """
    Делает срез по исходным данным по требуемым в задании параметрам.
    df: исходный датафрейм
    district: муниципалитет
    year: год
    target: пол, причина смерти или None
    Возвращает датафрейм формата входных данных.
    """
    
    if target is None:
        df_slice = (df[(df['Муниципалитет']==district)&(df['Год']==year)]
            .groupby(['Возрастная группа', 'Стандартный вес группы'], as_index=False)
            .agg({'Число умерших': 'sum', 'Численность населения': 'first'}))
    
    elif target in df['Причина смерти'].unique():
        df_slice = (df[(df['Муниципалитет']==district)&(df['Год']==year)&(df['Причина смерти']==target)]
            .groupby(['Возрастная группа', 'Стандартный вес группы'], as_index=False)
            .agg({'Число умерших': 'sum', 'Численность населения': 'first'}))
    
    elif target in df['Пол'].unique():
        df_slice = (df[(df['Муниципалитет']==district)&(df['Год']==year)&(df['Пол']==target)]
            .groupby(['Возрастная группа', 'Стандартный вес группы'], as_index=False)
            .agg({'Число умерших': 'sum', 'Численность населения': 'first'}))
    
    else:
        raise ValueError("Неверное значение target, выполнение прервано")
        
    return df_slice

In [8]:
def SCR_SE_calc(
    df: pd.DataFrame):

    """
    Рассчитывает стандартизованный коэффициент смертности и стандартную ошибку для него по датафрейму формата исходных данных.
    df: датафрейм формата исходных данных
    Возвращает стандартизованный коэффициент смертности и стандартную ошибку.
    """
    
    # Создание колонки 'Коэффициент смертности'.
    df['Коэффициент смертности'] = df['Число умерших'] / df['Численность населения']
    # Вычисление стандартизованного коэффициента смертности для среза.
    SCR = (df['Коэффициент смертности'] * df['Стандартный вес группы']).sum() * 100000
    # Вычисление стандартной ошибки стандартизованного коэффициента смертности для среза (пуассоновская аппроксимация).
    SE = ((df['Стандартный вес группы'].pow(2) * df['Коэффициент смертности'] / df['Численность населения']).sum() ** 0.5) * 100000
    return SCR.item(), SE.item()

In [9]:
def SCR_SE_by_slice(
    df: pd.DataFrame,
    district: str,
    year: int,
    target: str = None):

    """
    Расчитывает стандартизованный коэффициент смертности и его стандартную посредством функций get_slice и SCR_SE_calc.
    df: исходный датафрейм
    district: муниципалитет
    year: год
    target: пол, причина смерти или None
    Возвращает стандартизованный коэффициент смертности и стандартную ошибку.
    """
    
    # Создание таблицы среза через вызов функции get_slice.
    df_slice = get_slice(df, district, year, target)
    return SCR_SE_calc(df_slice)

## Вычисление

In [10]:
# Создание набора данных из всех комбинаций всех уникальных значений колонок 'Муниципалитет' и 'Год'.
targets_district_year_list = [[x, y] for x in df['Муниципалитет'].unique() for y in df['Год'].unique()]

# Создание набора данных из всех комбинаций всех уникальных значений колонок 'Муниципалитет', 'Год' и 'Пол'.
targets_district_year_gender_list = [[x, y, z] for x in df['Муниципалитет'].unique() \
                                    for y in df['Год'].unique() for z in df['Пол'].unique()]

# Создание набора данных из всех комбинаций всех уникальных значений колонок 'Муниципалитет', 'Год' и 'Причина смерти'.
targets_district_year_death_cause_list = [[x, y, z] for x in df['Муниципалитет'].unique() \
                                          for y in df['Год'].unique() for z in df['Причина смерти'].unique()]

In [11]:
# Названия колонок будующей таблицы с суммарным разрезом и разрезами по гендеру и причинам смерти.
columns_list = ['Муниципалитет', 'Год', 'Пол', 'Причина смерти', 'СКС', 'Стандартная ошибка']

In [12]:
# Суммарный разрез (объединяет данные по гендеру и причинам смерти сохраняя муниципалитет и год).
# Расчёт значений СКС и стандартной ошибки и включение этих данных в содержимое листа result_district_year_list
result_district_year_list = [row + ['Все', 'Все'] + list(SCR_SE_by_slice(df, row[0], row[1])) \
                             for row in targets_district_year_list]
# Создание датафрейма
df_result_district_year = pd.DataFrame(result_district_year_list, columns=columns_list)

# Разрез по гендеру (объединяет данные по гендеру сохраняя муниципалитет, год и причины смерти)
# Расчёт значений СКС и стандартной ошибки и включение этих данных в содержимое листа result_district_year_gender_list
result_district_year_gender_list = [row + ['Все'] + list(SCR_SE_by_slice(df, row[0], row[1], row[2])) \
                             for row in targets_district_year_gender_list]
# Создание датафрейма
df_result_district_year_gender = pd.DataFrame(result_district_year_gender_list, columns=columns_list)

# Разрез по причинам смерти (объединяет данные по причинам смерти сохраняя муниципалитет, год и гендер)
# Расчёт значений СКС и стандартной ошибки и включение этих данных в содержимое листа result_district_year_death_cause_list
result_district_year_death_cause_list = [[row[0], row[1], 'Все', row[2]] + list(SCR_SE_by_slice(df, row[0], row[1], row[2])) \
                             for row in targets_district_year_death_cause_list]
# Создание датафрейма
df_result_district_year_death_cause = pd.DataFrame(result_district_year_death_cause_list, columns=columns_list)

# Создание общего датафрейма
df_result = pd.concat([df_result_district_year, df_result_district_year_gender, df_result_district_year_death_cause], ignore_index=True)

In [13]:
df_result.head()

,Муниципалитет,Год,Пол,Причина смерти,СКС,Стандартная ошибка
0,Район 1,2017,Все,Все,4101.024238,1826.281171
1,Район 1,2018,Все,Все,1662.688463,595.858100
2,Район 1,2019,Все,Все,1009.803697,418.190239
3,Район 1,2020,Все,Все,3375.966423,888.348969
4,Район 1,2021,Все,Все,1341.353038,297.584294


# Интерпретация

## Константы

In [14]:
# Определение уровня значимости, который будет использоваться для расчётов.
SL = 0.05

## Функции

### Статика

In [15]:
def significance(
    SCR: float,
    SE: float,
    SL: float = 0.05):

    """
    Расчёт значимости отличия значения стандартизованного коэффициента смертности от нуля.
    SCR: стандартизованный коэффициент смертности
    SE: стандартная ошибка
    SL: уровень значимости
    Возвращает значимость в формате "Да" или "Нет".
    """
    
    # Обработка случая, когда стандартизованный коэффициент смертности равен нулю.
    if SCR == 0:
        return 'Нет'
    # Сравнение отношения стандартизованного коэффициента смертности к его стандартной ошибке с критическим значением z-статистики.
    if SCR/SE > stats.norm.ppf(1 - SL/2):
        return 'Да'
    else:
        return 'Нет'

In [16]:
def significance_column(df: pd.DataFrame,
                        SL: float = 0.05):

    """
    Добавляет к таблице формата df_result колонку "Значимость (p < x)" (значимость отличия значения СКС от нуля),
        содержащую значения "Да" или "Нет", где x - уровень значимости.
    df: датафрейм формата df_result
    SL: уровень значимости
    """
    # Добавление колонки 'Значимость (p < x)', где x - уровень значимости, через вызов функции significance.
    df['Значимость СКС (p < {})'.format(SL)] = (df[['СКС', 'Стандартная ошибка']]
        .apply(lambda x: significance(x['СКС'], x['Стандартная ошибка'], SL), axis=1))

In [17]:
def z_test(
    SCR_1: float,
    SE_1: float,
    SCR_2: float,
    SE_2: float,
    SL: float = 0.05):

    """
    Рассчитывает значимость различия между двумя значениями стандартизованного коэффициента смертности и показывает какое из них больше.
    Если +1, то первое, если -1, то второе.
    SCR_1: первый стандартизованный коэффициент смертности
    SE_1: стандартная ошибка для первого стандартизованного коэффициента смертности
    SCR_2: второй стандартизованный коэффициент смертности
    SE_2: стандартная ошибка для второго стандартизованного коэффициента смертности
    SL: уровень значимости
    Возвращает статистически значимое отличие SCR_1 от SCR_2 в формате '>', '=', '<'.
    """
    
    # Проверка на равенство нулю обоих стандартизованных коэффициентов смертности.
    if SCR_1 == 0 and SCR_2 == 0:
        return '='
    # Вычисление z-критерия для пары стандартизованных коэффициентов смертности.
    Z = (SCR_1 - SCR_2)/math.sqrt(SE_1**2 + SE_2**2)
    # Вычисление z-критерия для заданного уровня значимости (z-статистики).
    z = stats.norm.ppf(1 - SL/2)
    # Выбор результата по значениям Z и z.
    if abs(Z) - z > 0 and SCR_1 - SCR_2 > 0:
        z_result = '>'
    elif abs(Z) - z > 0 and SCR_1 - SCR_2 < 0:
        z_result = '<'
    else:
        z_result = '='
    return z_result

### Динамика

In [18]:
def weighted_regression(
    df: pd.DataFrame,
    district: str,
    gender: str,
    death_cause: str,
    SL: float = 0.05):
    """
    Рассчитывает взвешенную линейную регрессию для среза по требуемым параметрам муниципалитета, пола и причины смерти.
    df: датафрейм формата df_result
    district: требуемый муниципалитет
    gender: требуемый пол
    death_cause: требуемая причина смерти
    SL: уровень значимости
    Возвращает угловой коэффициент регрессии и его значимость в формате 'Да' или 'Нет'.
    """
    # получение среза
    df_slice = df[(df['Муниципалитет']==district)&(df['Пол']==gender)&(df['Причина смерти']==death_cause)&(df['Стандартная ошибка']!=0)]
    # получение количества точек
    n_elements = df_slice.shape[0]
    if n_elements < 4:
        return None, 'В группе недостаточно случаев'
    # получение вектора годов (ось x)
    x = np.array(df_slice['Год'])
    # получение вектора стандартизованных коэффициентов смертности (ось y)
    y = np.array(df_slice['СКС'])
    # получение вектора весов
    weights = 1 / np.array(df_slice['Стандартная ошибка'])**2

    # получение суммы весов
    weights_sum = weights.sum()
    # получение взвешенного среднего для годов (ось x)
    x_weights = np.sum(weights * x) / weights_sum
    # получение взвешенного среднего для стандартизованных коэффициентов смертности (ось y)
    y_weights = np.sum(weights * y) / weights_sum
    
    # получение числителя для углового коеффициента регрессии (b)
    b_numerator = np.sum(weights * (x - x_weights) * (y - y_weights))
    # получение знаменателя для углового коеффициента регрессии (b)
    b_denominator = np.sum(weights * (x - x_weights)**2)
    # защита от деления на 0
    if b_denominator == 0:
        raise ValueError("Нулевой знаменатель — невозможно оценить тренд")
    # получение углового коеффициента регрессии (b)
    b = b_numerator / b_denominator
    # получение сдвига регрессии (a)
    a = y_weights - b * x_weights
    
    # получение остатков (которые дальше будут минимизироваться)
    residuals = y - (a + b * x)
    # получение взвешенной дисперсии остатков
    sigma2 = np.sum(weights * residuals**2) / (n_elements - 2)
    # получение стандартной ошибки углового коэффициента регрессии (b)
    se_b = np.sqrt(sigma2 / b_denominator)
    
    # деление углового коэффициента регрессии на его стандартную ошибку (нужно для оценки его значимости)
    t_stat = b / se_b
    # получение значения функции распределения Стьюдента (возвращает вероятность того, что случайная величина меньше модуля t_stat)
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n_elements-2))

    # проверка значимости и вывод
    if p_value < SL:
        return b, 'Да'
    else:
        return b, 'Нет'

## Вычисление

### Статика

In [19]:
# Добавление колонки "Значимость СКС (p < x)" где x - требуемый уровень значимости.
# Показывает статистическую значимость отличия значения стандартизованного коэффициента смертности от нуля.
significance_column(df_result)

In [20]:
df_result.head()

,Муниципалитет,Год,Пол,Причина смерти,СКС,Стандартная ошибка,Значимость СКС (p < 0.05)
0,Район 1,2017,Все,Все,4101.024238,1826.281171,Да
1,Район 1,2018,Все,Все,1662.688463,595.858100,Да
2,Район 1,2019,Все,Все,1009.803697,418.190239,Да
3,Район 1,2020,Все,Все,3375.966423,888.348969,Да
4,Район 1,2021,Все,Все,1341.353038,297.584294,Да


In [21]:
# Создание таблицы, где муниципалитет, год, пол и причина смерти войдут в мультииндекс.
df_result_indexed = df_result.set_index(['Муниципалитет', 'Год', 'Пол', 'Причина смерти'])

### Статика между муниципалитетами

In [22]:
# Идея: попарно сравнить различия в стандартизованных коэффициентов смертности между разными муниципалитетами
# для всех комбинаций остальных параметров.

In [23]:
# Названия колонок будующей таблицы с попарным сравнением стандартизованных коэффициентов смертности
# между муниципалитетами для всех комбинаций остальных параметров.
columns_comparison_out_list = ['Муниципалитет 1', 'Муниципалитет 2', 'Год', 'Пол', 'Причина смерти',  'Значимое отличие (p < {})'.format(SL)]

In [24]:
# Создание набора ключей для срезов по двум разным муниципалитетам и гендерам
# для сопоставления стандартизованных коэффициентов смерти по годам.
comparision_out_district_gender_list = [[x, y, z, k, 'Все']
                                        for x in df_result['Муниципалитет'].unique()
                                        for y in df_result['Муниципалитет'].unique()
                                        for z in df_result['Год'].unique()
                                        for k in df_result['Пол'].unique()
                                        if  y < x]

# Создание набора ключей для срезов по двум разным муниципалитетам и причинам смерти
# для сопоставления стандартизованных коэффициентов смерти по годам.
comparision_out_district_death_cause_list = [[x, y, z, 'Все', l]
                                             for x in df_result['Муниципалитет'].unique()
                                             for y in df_result['Муниципалитет'].unique()
                                             for z in df_result['Год'].unique()
                                             for l in df_result['Причина смерти'].unique()
                                             if  y < x and l != 'Все']

# Суммарный набор ключей по двум разным срезам для сопоставления стандартизованных коэффициентов смерти по годам.
comparison_out_list = comparision_out_district_gender_list + comparision_out_district_death_cause_list

# Добавление данных стандартизованных коэффициентов смерти и стандартных ошибок для них.
comparison_out_list = [row + df_result_indexed.loc[(row[0], row[2], row[3], row[4])][['СКС', 'Стандартная ошибка']].tolist() \
                   + df_result_indexed.loc[(row[1], row[2], row[3], row[4])][['СКС', 'Стандартная ошибка']].tolist()
                        for row in comparison_out_list]

In [25]:
# Сопоставление стандартизованных коэффициентов смерти по годам, и включение этих данных в содержимое листа result_comparison_list
result_comparison_out_list = [row[:5] + list(z_test(row[5], row[6], row[7], row[8])) \
                             for row in comparison_out_list]
# Создание датафрейма
df_result_comparison_out = (pd.DataFrame(result_comparison_out_list, columns=columns_comparison_out_list)
    .sort_values(['Муниципалитет 1', 'Муниципалитет 2', 'Пол', 'Причина смерти', 'Год']))

In [26]:
df_result_comparison_out

,Муниципалитет 1,Муниципалитет 2,Год,Пол,Причина смерти,Значимое отличие (p < 0.05)
63,Район 2,Район 1,2017,Все,Болезни сердца,=
67,Район 2,Район 1,2018,Все,Болезни сердца,=
71,Район 2,Район 1,2019,Все,Болезни сердца,=
75,Район 2,Район 1,2020,Все,Болезни сердца,=
79,Район 2,Район 1,2021,Все,Болезни сердца,=
...,...,...,...,...,...,...
50,Район 3,Район 2,2019,Мужчины,Все,=
53,Район 3,Район 2,2020,Мужчины,Все,=
56,Район 3,Район 2,2021,Мужчины,Все,=
59,Район 3,Район 2,2022,Мужчины,Все,=


In [27]:
# Словарь замен для расчёта направленности общего различия между группами.
p_dict = {'=': 0, '>': 1, '<': -1}

In [28]:
# Создание колонки вида "Значимое отличие (p < 0.05) int"
# (она не была сгенерирована сразу т.к. знаки '>', '=', '<' более наглядные).
df_result_comparison_out['Значимое отличие (p < {} int)'.format(SL)] = (df_result_comparison_out['Значимое отличие (p < {})'.format(SL)]
    .apply(lambda x: p_dict[x]))
# Суммирование по колонке "Значимое отличие (p < 0.05) int" внутри каждой группы.
(df_result_comparison_out.groupby(['Муниципалитет 1', 'Муниципалитет 2', 'Пол', 'Причина смерти'], as_index=False)
    ['Значимое отличие (p < {} int)'.format(SL)].sum())

,Муниципалитет 1,Муниципалитет 2,Пол,Причина смерти,Значимое отличие (p < 0.05 int)
0,Район 2,Район 1,Все,Болезни сердца,0
1,Район 2,Район 1,Все,Все,0
2,Район 2,Район 1,Все,Другие болезни органов дыхания,0
3,Район 2,Район 1,Все,Другие болезни системы кровообращения,0
4,Район 2,Район 1,Все,Острые респираторные заболевания,0
5,Район 2,Район 1,Женщины,Все,1
6,Район 2,Район 1,Мужчины,Все,0
7,Район 3,Район 1,Все,Болезни сердца,-3
8,Район 3,Район 1,Все,Все,-2
9,Район 3,Район 1,Все,Другие болезни органов дыхания,1


In [29]:
# Короткий вывод:
# 1) район 1 и Район 2 мало отличаются по структуре смертности
# 2) в Районе 3 смертность ото всех причин и от болезней сердца в частности ниже чем в Районе 1 и Районе 2,
# но в тоже самое время смертность от остальных причин в Районе 3 выше

### Статика внутри муниципалитетов

In [30]:
# Идея: попарно сравнить различия в стандартизованных коэффициентов смертности между гендерами и причинами смерти
# для всех комбинаций остальных параметров.

In [31]:
# Названия колонок будующей таблицы с попарным сравнением стандартизованных коэффициентов смертности по годам
# внутри муниципалитета для двух разных гендеров.
columns_comparison_in_gender_list = ['Муниципалитет', 'Год', 'Пол 1', 'Пол 2', 'Причина смерти',  'Значимое отличие (p < {})'.format(SL)]


# Колонки будующей таблицы с попарным сравнением стандартизованных коэффициентов смертности по годам
# внутри муниципалитета для всех попарных кимбинаций причин смерти.
columns_comparison_in_death_cause_list = ['Муниципалитет', 'Год', 'Пол', 'Причина смерти 1', \
                                          'Причина смерти 2',  'Значимое отличие (p < {})'.format(SL)]

In [32]:
# Создание набора ключей для срезов df_result по двум разным гендерам внутри каждого муниципалитета.
comparison_in_district_gender_list = [[x, y, 'Мужчины', 'Женщины', 'Все']
                                      for x in df_result['Муниципалитет'].unique()
                                      for y in df_result['Год'].unique()]

# Создание набора ключей для срезов df_result по двум разным причинам смерти внутри каждого муниципалитета.
comparison_in_district_death_cause_list = [[x, y, 'Все', k, l] for x in df_result['Муниципалитет'].unique() 
                                           for y in df_result['Год'].unique()
                                           for k in df_result['Причина смерти'].unique()
                                           for l in df_result['Причина смерти'].unique()
                                           if  k<l and k!='Все' and l!='Все']

# Добавление данных стандартизованных коэффициентов смерти и стандартных ошибок
# для них для срезов по двум разным гендерам внутри муниципалитета.
comparison_in_gender_list = [row + df_result_indexed.loc[(row[0], row[1], row[2], row[4])][['СКС', 'Стандартная ошибка']].tolist() \
                   + df_result_indexed.loc[(row[0], row[1], row[3], row[4])][['СКС', 'Стандартная ошибка']].tolist()
                           for row in comparison_in_district_gender_list]

# Добавление данных стандартизованных коэффициентов смерти и стандартных ошибокдля них
# для срезов по двум разным причинам смерти внутри муниципалитета.
comparison_in_death_cause_list = [row + df_result_indexed.loc[(row[0], row[1], row[2], row[3])][['СКС', 'Стандартная ошибка']].tolist() \
                   + df_result_indexed.loc[(row[0], row[1], row[2], row[4])][['СКС', 'Стандартная ошибка']].tolist()
                                   for row in comparison_in_district_death_cause_list]

In [33]:
# Сопоставление стандартизованных коэффициентов смерти для срезов по двум разным гендерам.
result_comparison_in_gender_list = [row[:5] + list(z_test(row[5], row[6], row[7], row[8])) \
                             for row in comparison_in_gender_list]

# Создание датафрейма для срезов по двум разным гендерам.
df_result_comparison_in_gender = (pd.DataFrame(result_comparison_in_gender_list, columns=columns_comparison_in_gender_list)
    .sort_values(['Муниципалитет', 'Пол 1', 'Пол 2', 'Причина смерти', 'Год']))

# Сопоставление стандартизованных коэффициентов смерти для срезов по двум разным причинам смерти.
result_comparison_in_death_cause_list = [row[:5] + list(z_test(row[5], row[6], row[7], row[8])) \
                             for row in comparison_in_death_cause_list]

# Создание датафрейма для срезов по двум разным причинам смерти.
df_result_comparison_in_death_cause = (pd.DataFrame(result_comparison_in_death_cause_list, \
                                                    columns=columns_comparison_in_death_cause_list)
    .sort_values(['Муниципалитет', 'Пол', 'Причина смерти 1', 'Причина смерти 2', 'Год']))

In [34]:
df_result_comparison_in_gender

,Муниципалитет,Год,Пол 1,Пол 2,Причина смерти,Значимое отличие (p < 0.05)
0,Район 1,2017,Мужчины,Женщины,Все,=
1,Район 1,2018,Мужчины,Женщины,Все,=
2,Район 1,2019,Мужчины,Женщины,Все,=
3,Район 1,2020,Мужчины,Женщины,Все,=
4,Район 1,2021,Мужчины,Женщины,Все,=
5,Район 1,2022,Мужчины,Женщины,Все,=
6,Район 1,2023,Мужчины,Женщины,Все,>
7,Район 2,2017,Мужчины,Женщины,Все,=
8,Район 2,2018,Мужчины,Женщины,Все,=
9,Район 2,2019,Мужчины,Женщины,Все,=


In [35]:
df_result_comparison_in_death_cause

,Муниципалитет,Год,Пол,Причина смерти 1,Причина смерти 2,Значимое отличие (p < 0.05)
0,Район 1,2017,Все,Болезни сердца,Другие болезни органов дыхания,=
6,Район 1,2018,Все,Болезни сердца,Другие болезни органов дыхания,>
12,Район 1,2019,Все,Болезни сердца,Другие болезни органов дыхания,>
18,Район 1,2020,Все,Болезни сердца,Другие болезни органов дыхания,>
24,Район 1,2021,Все,Болезни сердца,Другие болезни органов дыхания,>
...,...,...,...,...,...,...
101,Район 3,2019,Все,Другие болезни системы кровообращения,Острые респираторные заболевания,=
107,Район 3,2020,Все,Другие болезни системы кровообращения,Острые респираторные заболевания,=
113,Район 3,2021,Все,Другие болезни системы кровообращения,Острые респираторные заболевания,=
119,Район 3,2022,Все,Другие болезни системы кровообращения,Острые респираторные заболевания,<


In [36]:
# Создание колонки вида "Значимое отличие (p < 0.05) int"
# (она не была сгенерирована сразу т.к. знаки '>', '=', '<' более наглядные).
df_result_comparison_in_death_cause['Значимое отличие (p < {} int)'.format(SL)] = \
(df_result_comparison_in_death_cause['Значимое отличие (p < {})'.format(SL)]
    .apply(lambda x: p_dict[x]))
# Суммирование по колонке "Значимое отличие (p < 0.05) int" внутри каждой группы.
(df_result_comparison_in_death_cause.groupby(['Муниципалитет', 'Пол', 'Причина смерти 1', 'Причина смерти 2'], as_index=False)
    ['Значимое отличие (p < {} int)'.format(SL)].sum())

,Муниципалитет,Пол,Причина смерти 1,Причина смерти 2,Значимое отличие (p < 0.05 int)
0,Район 1,Все,Болезни сердца,Другие болезни органов дыхания,6
1,Район 1,Все,Болезни сердца,Другие болезни системы кровообращения,7
2,Район 1,Все,Болезни сердца,Острые респираторные заболевания,7
3,Район 1,Все,Другие болезни органов дыхания,Другие болезни системы кровообращения,0
4,Район 1,Все,Другие болезни органов дыхания,Острые респираторные заболевания,0
5,Район 1,Все,Другие болезни системы кровообращения,Острые респираторные заболевания,0
6,Район 2,Все,Болезни сердца,Другие болезни органов дыхания,7
7,Район 2,Все,Болезни сердца,Другие болезни системы кровообращения,6
8,Район 2,Все,Болезни сердца,Острые респираторные заболевания,7
9,Район 2,Все,Другие болезни органов дыхания,Другие болезни системы кровообращения,0


In [37]:
# Короткий вывод:
# 1) в Районе 3 эпизодически мужская смертность статистически значимо превышает женскую
# 2) болезни сердца статистически значимо преобладают над остальными причинами смерти во всех муниципалитетах

### Динамика

In [38]:
# Идея: установить наличие устойчивого временного тренда в группе,
# показав статистечески значимое отличие от нуля углового коэффициента регрессии b.

In [39]:
columns_regression_list = ['Муниципалитет', 'Пол', 'Причина смерти', 'Коэффициента регрессии (b)', 'Значимость (p < {})'.format(SL)]

In [40]:
# Создание набора ключей для срезов по муниципалитетам для вычисления взвешенной линейной регрессии.
regression_district_list = [[x, 'Все', 'Все']
                            for x in df_result['Муниципалитет'].unique()]

# Создание набора ключей для срезов по муниципалитетам и гендерам для вычисления взвешенной линейной регрессии.
regression_district_gender_list = [[x, y, 'Все']
                                   for x in df_result['Муниципалитет'].unique()
                                   for y in ['Мужчины', 'Женщины']]

# Создание набора ключей для срезов по муниципалитетам и причинам смерти для вычисления взвешенной линейной регрессии.
regression_district_death_cause_list = [[x, 'Все', z]
                                        for x in df_result['Муниципалитет'].unique()
                                        for z in df['Причина смерти'].unique()
                                        if z!='Все']

# Суммарный набор ключей по срезам для вычисления взвешенной линейной регрессии.
regression_list = regression_district_list + regression_district_gender_list + regression_district_death_cause_list

In [41]:
# Расчёт значений углового коэффициента взвешенной линейной регрессии и статистической значимости его отличия от 0,
# и включение этих данных в содержимое листа result_regression_list.
result_regression_list = [row + list(weighted_regression(df_result, row[0], row[1], row[2])) \
                             for row in regression_list]

# Создание датафрейма на основе листа regression_list.
df_result_regression = (pd.DataFrame(result_regression_list, columns=columns_regression_list)
    .sort_values(['Муниципалитет', 'Пол', 'Причина смерти'], ignore_index=True))

In [42]:
df_result_regression

,Муниципалитет,Пол,Причина смерти,Коэффициента регрессии (b),Значимость (p < 0.05)
0,Район 1,Все,Болезни сердца,10.583881,Нет
1,Район 1,Все,Все,25.757679,Нет
2,Район 1,Все,Другие болезни органов дыхания,9.435371,Нет
3,Район 1,Все,Другие болезни системы кровообращения,NaN,В группе недостаточно случаев
4,Район 1,Все,Острые респираторные заболевания,NaN,В группе недостаточно случаев
5,Район 1,Женщины,Все,-88.480430,Нет
6,Район 1,Мужчины,Все,92.817444,Нет
7,Район 2,Все,Болезни сердца,170.904892,Нет
8,Район 2,Все,Все,154.027832,Нет
9,Район 2,Все,Другие болезни органов дыхания,NaN,В группе недостаточно случаев


In [43]:
# Короткий вывод:
# 1) в Районе 2 статистически значимо обнаружен рост мужской смертности в исследуемом диапазоне дат
# 2) в Районе 3 статистически значимо обнаружено снижение смертности от болезней сердца в исследуемом диапазоне дат